# Smart Agriculture Monitoring — Irrigation Prediction (Google Colab)

**Aim:** Predict whether a crop needs irrigation, from sensor readings.

**How to run:** Open this notebook in Google Colab and press **Runtime → Run all**.
When the **Choose Files** button appears, pick `agriculture_sensor_data.csv`
(the CSV file included with this project). The rest runs by itself — no
libraries to install, pandas and scikit-learn are already in Colab.

**Note on the upload button:** If it shows "No file chosen" and says the
widget is only available in the current browser session, just **rerun that
cell** — Colab only enables the button after the cell runs. That is normal
Colab behaviour, not an error in the notebook.

In [ ]:
# Upload: choose the CSV file
import io

import pandas as pd

df = None
try:
    from google.colab import files
    uploaded = files.upload()  # click "Choose Files", pick agriculture_sensor_data.csv
    if uploaded:
        name = next(iter(uploaded))
        df = pd.read_csv(io.BytesIO(uploaded[name]))
        print("Loaded uploaded file:", name)
except ImportError:
    pass  # not running in Colab, use the local file below

if df is None:
    df = pd.read_csv("agriculture_sensor_data.csv")
    print("Loaded local file: agriculture_sensor_data.csv")

Loaded local file: agriculture_sensor_data.csv


## Investigation 1: Understanding Dataset

In [ ]:
# Investigation 1: Understanding Dataset
print(df.columns)
print("\nTarget Variable: irrigation_needed (1 = irrigate, 0 = no need)")
print(df["irrigation_needed"].value_counts())
print("\nFeatures:", [c for c in df.columns if c != "irrigation_needed"])
print("\nDataset shape:", df.shape)
print(df.head())

Index(['crop', 'soil_moisture', 'temperature', 'humidity', 'light',
       'irrigation_needed'],
      dtype='str')

Target Variable: irrigation_needed (1 = irrigate, 0 = no need)
irrigation_needed
0    187
1    113
Name: count, dtype: int64

Features: ['crop', 'soil_moisture', 'temperature', 'humidity', 'light']

Dataset shape: (300, 6)
     crop  soil_moisture  temperature  humidity  light  irrigation_needed
0  tomato           83.6         34.4      46.4    858                  0
1   onion           36.8         29.0      91.7    880                  1
2   wheat           30.1         25.6      74.5    240                  1
3  tomato           54.3         37.7      58.4    373                  0
4  tomato           89.0         17.8      47.7    515                  0


## Investigation 2: Preparing Data for Logistic Regression

In [ ]:
# Investigation 2: Preparing Data for Logistic Regression
from sklearn.model_selection import train_test_split

# One-Hot Encode the crop names (first crop kept as baseline)
df_enc = pd.get_dummies(df, columns=["crop"], drop_first=True)
X = df_enc.drop("irrigation_needed", axis=1)
y = df_enc["irrigation_needed"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)
print("Training Samples:", len(X_train))
print("Testing Samples:", len(X_test))
print("Number of Features:", X.shape[1])
print(X_train.head())

Training Samples: 240
Testing Samples: 60
Number of Features: 7
     soil_moisture  temperature  ...  crop_tomato  crop_wheat
232           64.0         19.3  ...        False       False
59            31.3         33.1  ...        False        True
6             67.0         15.3  ...        False        True
185           22.9         26.2  ...        False        True
173           39.6         27.1  ...        False       False

[5 rows x 7 columns]


## Investigation 3: Building and Training the Logistic Regression Model

In [ ]:
# Investigation 3: Building and Training the Logistic Regression Model
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

print("Intercept:", model.intercept_)
print("\nRegression Coefficients:")
for feature, coef in zip(X.columns, model.coef_[0]):
    print(f"{feature} : {coef}")

Intercept: [16.45223357]

Regression Coefficients:
soil_moisture : -0.3268127947009695
temperature : -0.018012796141655924
humidity : -0.02179635285939521
light : 0.0014504673703293546
crop_rice : 2.8697141131615553
crop_tomato : 0.41889001649644897
crop_wheat : -1.8285355193784678


## Investigation 4: Predicting Irrigation Need using the Logistic Regression Model

In [ ]:
# Investigation 4: Predicting Irrigation Need using the Logistic Regression Model
y_pred = model.predict(X_test)

results = pd.DataFrame(
    {"Actual Status": y_test.values, "Predicted Status": y_pred})
print(results.head(10))

# Predict for one new reading (built as a DataFrame, so feature names stay valid)
new_raw = pd.DataFrame([{
    "crop": "wheat", "soil_moisture": 28, "temperature": 34,
    "humidity": 55, "light": 950,
}])
new_X = pd.get_dummies(new_raw, columns=["crop"], drop_first=True)
new_X = new_X.reindex(columns=X.columns, fill_value=0)
print("\nPredicted Status:",
      ["YES - Irrigate now" if p == 1 else "NO - Soil is good"
       for p in model.predict(new_X)])

results["Correct Prediction"] = (
    results["Actual Status"] == results["Predicted Status"])
print(results.head(10))

   Actual Status  Predicted Status
0              1                 1
1              1                 1
2              0                 0
3              0                 0
4              0                 0
5              0                 0
6              0                 0
7              1                 1
8              1                 1
9              0                 0

Predicted Status: ['YES - Irrigate now']
   Actual Status  Predicted Status  Correct Prediction
0              1                 1                True
1              1                 1                True
2              0                 0                True
3              0                 0                True
4              0                 0                True
5              0                 0                True
6              0                 0                True
7              1                 1                True
8              1                 1                True
9              0       

## Investigation 5: Evaluating the Performance of the Logistic Regression Model

In [ ]:
# Investigation 5: Evaluating the Performance of the Logistic Regression Model
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print("Accuracy:", accuracy_score(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

Accuracy: 0.9666666666666667
[[35  1]
 [ 1 23]]
              precision    recall  f1-score   support

           0       0.97      0.97      0.97        36
           1       0.96      0.96      0.96        24

    accuracy                           0.97        60
   macro avg       0.97      0.97      0.97        60
weighted avg       0.97      0.97      0.97        60



## Investigation 6: Interpreting the Logistic Regression Model for Decision-Making

In [ ]:
# Investigation 6: Interpreting the Logistic Regression Model for Decision-Making
coef_df = pd.DataFrame(
    {"Feature": X.columns, "Coefficient": model.coef_[0]})
print(coef_df)
print("Intercept:", model.intercept_[0])
ranked = coef_df.assign(
    Absolute_Coefficient=coef_df["Coefficient"].abs()
).sort_values("Absolute_Coefficient", ascending=False)
print(ranked)

         Feature  Coefficient
0  soil_moisture    -0.326813
1    temperature    -0.018013
2       humidity    -0.021796
3          light     0.001450
4      crop_rice     2.869714
5    crop_tomato     0.418890
6     crop_wheat    -1.828536
Intercept: 16.452233573340088
         Feature  Coefficient  Absolute_Coefficient
4      crop_rice     2.869714              2.869714
6     crop_wheat    -1.828536              1.828536
5    crop_tomato     0.418890              0.418890
0  soil_moisture    -0.326813              0.326813
2       humidity    -0.021796              0.021796
1    temperature    -0.018013              0.018013
3          light     0.001450              0.001450


## Save the trained model

In [ ]:
# Save the trained model to a .pkl file
import pickle

with open("agriculture_model.pkl", "wb") as file:
    pickle.dump(model, file)
print("Model saved as agriculture_model.pkl")

Model saved as agriculture_model.pkl
